In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as pl

In [2]:
DATA_PATH = Path("../data")

# EDA

In [3]:
files = [
    "CGCS-Template.csv",
    "Q1-Graph1.csv",
    "Q1-Graph2.csv",
    "Q1-Graph3.csv",
    "Q1-Graph4.csv",
    "Q1-Graph5.csv",
    "Q2-Seed1.csv",
    "Q2-Seed2.csv",
    "Q2-Seed3.csv",
    "CGCS-GraphData-NodeTypes.csv",
    "CGCS-Template-NodeTypes.csv",
    "NodeTypeDescriptions.csv",
    "DemographicCategories.csv"
]

De los archivos presentes en el dataset, existen tanto los Core Graph Files como los Metadata Files. Para el análisis exploratorio de datos, se han utilizado ambos tipos de archivos, ya que los Metadata Files proporcionan información adicional sobre los nodos y las relaciones presentes en los Core Graph Files.

In [4]:
loaded = {}

for f in files:
  path = DATA_PATH / f
  if path.exists():
    try:
      df = pd.read_csv(path)
      loaded[f] = df
      print(f"{f}: {df.shape}")
    except Exception as e:
      print(f"{f}: ERROR -> {e}")
  else:
    print(f"{f}: NOT FOUND")

CGCS-Template.csv: (1325, 11)
Q1-Graph1.csv: (1216, 11)
Q1-Graph2.csv: (1300, 11)
Q1-Graph3.csv: (729, 11)
Q1-Graph4.csv: (732, 11)
Q1-Graph5.csv: (395, 11)
Q2-Seed1.csv: (1, 11)
Q2-Seed2.csv: (1, 11)
Q2-Seed3.csv: (1, 11)
CGCS-GraphData-NodeTypes.csv: (200912, 2)
CGCS-Template-NodeTypes.csv: (88, 2)
NodeTypeDescriptions.csv: (5, 3)
DemographicCategories.csv: (29, 2)


In [6]:
for name, df in loaded.items():
  print("="*60)
  print(name)
  print(df.head())
  print(df.columns.tolist())

CGCS-Template.csv
   Source  eType  Target    Time  Weight  SourceLocation  TargetLocation  \
0       0      4     -99     -99     -99             NaN             NaN   
1      41      0      34   86400       1             NaN             NaN   
2      37      0      27   94461       1             NaN             NaN   
3      34      1      27  107548       1             5.0             5.0   
4      41      0      37  127838       1             NaN             NaN   

   SourceLatitude  SourceLongitude  TargetLatitude  TargetLongitude  
0             NaN              NaN             NaN              NaN  
1             NaN              NaN             NaN              NaN  
2             NaN              NaN             NaN              NaN  
3             NaN              NaN             NaN              NaN  
4             NaN              NaN             NaN              NaN  
['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'So

Los tipos de columnas en los grafos presentes en el dataset son los siguientes:
['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']

Estos tipos de columnas se encuentran en los archivos que contienen Q como sufijo, es decir, en los Core Graph Files. Estas columnas representan información sobre las relaciones entre nodos, incluyendo el nodo de origen (Source), el tipo de relación (eType), el nodo de destino (Target), la marca de tiempo (Time), el peso de la relación (Weight), la ubicación del nodo de origen (SourceLocation), la ubicación del nodo de destino (TargetLocation), y las coordenadas geográficas tanto del nodo de origen como del nodo de destino (SourceLatitude, SourceLongitude, TargetLatitude, TargetLongitude).

Lo demás archivos, es decir, los Metadata Files, contienen información adicional sobre los nodos y las relaciones, pero no siguen el mismo formato que los Core Graph Files. Estos archivos pueden incluir columnas como 'id', 'type', 'name', 'description', entre otras, dependiendo del tipo de nodo o relación que se esté describiendo.

In [7]:
summary = []

for name, df in loaded.items():
    summary.append({
        "file": name,
        "rows": df.shape[0],
        "cols": df.shape[1],
        "columns": ", ".join(df.columns[:6])
    })

summary_df = pd.DataFrame(summary)
summary_df.sort_values("rows", ascending=False)

,file,rows,cols,columns
9,CGCS-GraphData-NodeTypes.csv,200912,2,"NodeID, NodeType"
0,CGCS-Template.csv,1325,11,"Source, eType, Target, Time, Weight, SourceLoc..."
2,Q1-Graph2.csv,1300,11,"Source, eType, Target, Time, Weight, SourceLoc..."
1,Q1-Graph1.csv,1216,11,"Source, eType, Target, Time, Weight, SourceLoc..."
4,Q1-Graph4.csv,732,11,"Source, eType, Target, Time, Weight, SourceLoc..."
3,Q1-Graph3.csv,729,11,"Source, eType, Target, Time, Weight, SourceLoc..."
5,Q1-Graph5.csv,395,11,"Source, eType, Target, Time, Weight, SourceLoc..."
10,CGCS-Template-NodeTypes.csv,88,2,"NodeID, NodeType"
12,DemographicCategories.csv,29,2,"NodeID, Category"
11,NodeTypeDescriptions.csv,5,3,"NodeType, Description, Used in"


Los archivos Q1 corresponden a subgrafos candidatos de tamaño moderado, mientras que el template contiene un patrón pequeño de referencia. Los archivos NodeTypes proveen tipología de nodos necesaria para análisis estructural.

In [7]:
graph_files = [k for k in loaded if "Graph" in k or "Template" in k or "Seed" in k]
meta_files = [k for k in loaded if k not in graph_files]

print("Graph files:")
print(graph_files)

print("\nMetadata files:")
print(meta_files)

Graph files:
['CGCS-Template.csv', 'Q1-Graph1.csv', 'Q1-Graph2.csv', 'Q1-Graph3.csv', 'Q1-Graph4.csv', 'Q1-Graph5.csv', 'Q2-Seed1.csv', 'Q2-Seed2.csv', 'Q2-Seed3.csv', 'CGCS-GraphData-NodeTypes.csv', 'CGCS-Template-NodeTypes.csv']

Metadata files:
['NodeTypeDescriptions.csv', 'DemographicCategories.csv']


Metadata Files:
- NodeTypesDescriptions: Descripciones de tipos de nodos.
- DemographicCategories: Categorías demográficas (e.g., edad, género).

Core Graph Files:
Todos los demás

Se requieren las columnas `Source`, `eType`, `Target` para identificar las relaciones entre nodos, mientras que las columnas de ubicación y coordenadas permiten análisis geoespacial y la columna `Time` es crucial para análisis temporales.

In [8]:
expected_core = {"Source", "Target", "eType", "Time"}

for name, df in loaded.items():
  if any(x in name for x in ["Graph", "Template", "Seed"]):
    cols = set(df.columns)
    missing = expected_core - cols
    
    print("="*60)
    print(name)
    print("Columns:", list(df.columns))
    print("Missing required:", missing if missing else "None")

CGCS-Template.csv
Columns: ['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']
Missing required: None
Q1-Graph1.csv
Columns: ['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']
Missing required: None
Q1-Graph2.csv
Columns: ['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']
Missing required: None
Q1-Graph3.csv
Columns: ['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']
Missing required: None
Q1-Graph4.csv
Columns: ['Source', 'eType', 'Target', 'Time', 'Weight', 'SourceLocation', 'TargetLocation', 'SourceLatitude', 'SourceLongitude', 'TargetLatitude', 'TargetLongitude']

Como se puede evidenciar, todos los archivos menos los templates tienen las columnas mencionadas anteriormente, lo que indica que todo esta en orden. 

In [9]:
for name, df in loaded.items():
  if any(x in name for x in ["Graph", "Template", "Seed"]):
    print("="*60)
    print(name)
    print(df.dtypes)

CGCS-Template.csv
Source               int64
eType                int64
Target               int64
Time                 int64
Weight               int64
SourceLocation     float64
TargetLocation     float64
SourceLatitude     float64
SourceLongitude    float64
TargetLatitude     float64
TargetLongitude    float64
dtype: object
Q1-Graph1.csv
Source               int64
eType                int64
Target               int64
Time                 int64
Weight             float64
SourceLocation     float64
TargetLocation     float64
SourceLatitude     float64
SourceLongitude    float64
TargetLatitude     float64
TargetLongitude    float64
dtype: object
Q1-Graph2.csv
Source               int64
eType                int64
Target               int64
Time                 int64
Weight             float64
SourceLocation     float64
TargetLocation     float64
SourceLatitude     float64
SourceLongitude    float64
TargetLatitude     float64
TargetLongitude    float64
dtype: object
Q1-Graph3.csv
Source 

Solo valores conectores y en los NodesTypes se encuentra la columna `id`, que es un identificador único para cada nodo, lo cual es esencial para construir el grafo correctamente. En los demás archivos, la ausencia de esta columna no afecta la capacidad de análisis, ya que se centran en las relaciones y características de los nodos en lugar de su identificación única.

In [10]:
for name, df in loaded.items():
  na = df.isna().sum()
  na = na[na > 0]
  
  if len(na) > 0:
    print("="*60)
    print(name)
    print(na)

CGCS-Template.csv
SourceLocation     1024
TargetLocation     1024
SourceLatitude     1325
SourceLongitude    1325
TargetLatitude     1325
TargetLongitude    1325
dtype: int64
Q1-Graph1.csv
SourceLocation     1048
TargetLocation     1048
SourceLatitude     1048
SourceLongitude    1048
TargetLatitude     1048
TargetLongitude    1048
dtype: int64
Q1-Graph2.csv
SourceLocation     1099
TargetLocation     1099
SourceLatitude     1099
SourceLongitude    1099
TargetLatitude     1099
TargetLongitude    1099
dtype: int64
Q1-Graph3.csv
SourceLocation     641
TargetLocation     641
SourceLatitude     641
SourceLongitude    641
TargetLatitude     641
TargetLongitude    641
dtype: int64
Q1-Graph4.csv
SourceLocation     556
TargetLocation     556
SourceLatitude     556
SourceLongitude    556
TargetLatitude     556
TargetLongitude    556
dtype: int64
Q1-Graph5.csv
SourceLocation     271
TargetLocation     271
SourceLatitude     271
SourceLongitude    271
TargetLatitude     271
TargetLongitude    271
d

In [11]:
for name, df in loaded.items():
  dup = df.duplicated().sum()
  print(name, "duplicates:", dup)

CGCS-Template.csv duplicates: 0
Q1-Graph1.csv duplicates: 0
Q1-Graph2.csv duplicates: 0
Q1-Graph3.csv duplicates: 0
Q1-Graph4.csv duplicates: 92
Q1-Graph5.csv duplicates: 88
Q2-Seed1.csv duplicates: 0
Q2-Seed2.csv duplicates: 0
Q2-Seed3.csv duplicates: 0
CGCS-GraphData-NodeTypes.csv duplicates: 0
CGCS-Template-NodeTypes.csv duplicates: 0
NodeTypeDescriptions.csv duplicates: 0
DemographicCategories.csv duplicates: 0


In [12]:
for name, df in loaded.items():
  if "eType" in df.columns:
    print(name, sorted(df["eType"].dropna().unique()))

CGCS-Template.csv [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Q1-Graph1.csv [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Q1-Graph2.csv [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Q1-Graph3.csv [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Q1-Graph4.csv [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6)]
Q1-Graph5.csv [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6)]
Q2-Seed1.csv [np.int64(4)]
Q2-Seed2.csv [np.int64(4)]
Q2-Seed3.csv [np.int64(2)]


In [13]:
for name, df in loaded.items():
  if "Time" in df.columns:
    print("="*50)
    print(name)
    print("min:", df["Time"].min())
    print("max:", df["Time"].max())

CGCS-Template.csv
min: -99
max: 31536000
Q1-Graph1.csv
min: -662041253
max: 31536000
Q1-Graph2.csv
min: -732727926
max: 31536000
Q1-Graph3.csv
min: -209742644
max: 31536000
Q1-Graph4.csv
min: 98586
max: 31536000
Q1-Graph5.csv
min: 96346
max: 31536000
Q2-Seed1.csv
min: -685755382
max: -685755382
Q2-Seed2.csv
min: -623491200
max: -623491200
Q2-Seed3.csv
min: 1991785
max: 1991785


In [14]:
for name, df in loaded.items():
  if "Weight" in df.columns:
    neg = (df["Weight"] < 0).sum()
    zero = (df["Weight"] == 0).sum()
    
    print(name, "negative:", neg, "zero:", zero)

CGCS-Template.csv negative: 1 zero: 0
Q1-Graph1.csv negative: 0 zero: 0
Q1-Graph2.csv negative: 0 zero: 0
Q1-Graph3.csv negative: 0 zero: 0
Q1-Graph4.csv negative: 45 zero: 0
Q1-Graph5.csv negative: 30 zero: 15
Q2-Seed1.csv negative: 0 zero: 0
Q2-Seed2.csv negative: 0 zero: 0
Q2-Seed3.csv negative: 0 zero: 0


In [15]:
for name, df in loaded.items():
  if {"Source","Target"}.issubset(df.columns):
    self_loops = (df["Source"] == df["Target"]).sum()
    print(name, "self loops:", self_loops)

CGCS-Template.csv self loops: 0
Q1-Graph1.csv self loops: 0
Q1-Graph2.csv self loops: 0
Q1-Graph3.csv self loops: 0
Q1-Graph4.csv self loops: 4
Q1-Graph5.csv self loops: 0
Q2-Seed1.csv self loops: 0
Q2-Seed2.csv self loops: 0
Q2-Seed3.csv self loops: 0


In [16]:
quality_report = []

for name, df in loaded.items():
    quality_report.append({
        "file": name,
        "rows": len(df),
        "cols": len(df.columns),
        "missing_total": int(df.isna().sum().sum()),
        "duplicates": int(df.duplicated().sum())
    })

pd.DataFrame(quality_report)

,file,rows,cols,missing_total,duplicates
0,CGCS-Template.csv,1325,11,7348,0
1,Q1-Graph1.csv,1216,11,6288,0
2,Q1-Graph2.csv,1300,11,6594,0
3,Q1-Graph3.csv,729,11,3846,0
4,Q1-Graph4.csv,732,11,3336,92
5,Q1-Graph5.csv,395,11,1626,88
6,Q2-Seed1.csv,1,11,6,0
7,Q2-Seed2.csv,1,11,6,0
8,Q2-Seed3.csv,1,11,6,0
9,CGCS-GraphData-NodeTypes.csv,200912,2,0,0


**Resumen de Calidad de los Datos**: 
Todos los archivos de grafo contienen el esquema requerido (Fuente, Destino, tipo de arista, Tiempo).
No se detectaron problemas críticos de duplicación.
Los valores faltantes se concentran en atributos geográficos opcionales.
Los tipos de arista se encuentran dentro del rango esperado [0–6].

**Observaciones Temporales**: 
Las marcas de tiempo negativas corresponden a actividad de coautoría.
Las marcas de tiempo positivas representan interacciones del año 2025.
Los datos demográficos ocurren en una marca de tiempo fija (31536000).

**Observaciones Estructurales**: 
La plantilla incluye todos los tipos de interacción.
Algunos grafos candidatos carecen de tipos específicos de aristas:
- Q1-Graph4 y Q1-Graph5 carecen de interacciones de coautoría.
Los grafos semilla representan puntos de partida aislados.

In [17]:
def basic_graph_stats(df):
  nodes = set(df["Source"]) | set(df["Target"])
  n_nodes = len(nodes)
  n_edges = len(df)

  density = n_edges / (n_nodes * (n_nodes - 1)) if n_nodes > 1 else 0

  return {
    "nodes": n_nodes,
    "edges": n_edges,
    "density": density
  }

stats = []

for name, df in loaded.items():
  if {"Source", "Target"}.issubset(df.columns):
    s = basic_graph_stats(df)
    s["file"] = name
    stats.append(s)

pd.DataFrame(stats).sort_values("nodes", ascending=False)

,nodes,edges,density,file
1,93,1216,0.142122,Q1-Graph1.csv
0,88,1325,0.173067,CGCS-Template.csv
2,87,1300,0.173750,Q1-Graph2.csv
4,87,732,0.097835,Q1-Graph4.csv
5,86,395,0.054036,Q1-Graph5.csv
3,79,729,0.118306,Q1-Graph3.csv
6,2,1,0.500000,Q2-Seed1.csv
7,2,1,0.500000,Q2-Seed2.csv
8,2,1,0.500000,Q2-Seed3.csv


In [18]:
etype_dist = {}

for name, df in loaded.items():
  if ("Graph" in name or "Template" in name) and "eType" in df.columns:
    etype_dist[name] = df["eType"].value_counts(normalize=True)

etype_df = pd.DataFrame(etype_dist).fillna(0)
etype_df

,CGCS-Template.csv,Q1-Graph1.csv,Q1-Graph2.csv,Q1-Graph3.csv,Q1-Graph4.csv,Q1-Graph5.csv
eType,,,,,,
0,0.236981,0.153783,0.198462,0.149520,0.061475,0.043038
1,0.187925,0.107730,0.136154,0.069959,0.083333,0.035443
2,0.006792,0.005757,0.005385,0.008230,0.006831,0.027848
3,0.006792,0.005757,0.005385,0.008230,0.016393,0.101266
4,0.000755,0.000822,0.003077,0.001372,0.000000,0.000000
5,0.521509,0.695724,0.633077,0.711934,0.674863,0.513924
6,0.039245,0.030428,0.018462,0.050754,0.157104,0.278481


In [19]:
def degree_stats(df):
  out_deg = df["Source"].value_counts()
  in_deg = df["Target"].value_counts()

  total_deg = out_deg.add(in_deg, fill_value=0)

  return {
    "avg_degree": total_deg.mean(),
    "max_degree": total_deg.max(),
    "std_degree": total_deg.std()
  }

deg_stats = []

for name, df in loaded.items():
  if {"Source", "Target"}.issubset(df.columns):
    s = degree_stats(df)
    s["file"] = name
    deg_stats.append(s)

pd.DataFrame(deg_stats)

,avg_degree,max_degree,std_degree,file
0,30.113636,208.0,34.151263,CGCS-Template.csv
1,26.150538,135.0,21.401415,Q1-Graph1.csv
2,29.885057,192.0,27.735640,Q1-Graph2.csv
3,18.455696,68.0,14.815790,Q1-Graph3.csv
4,16.827586,64.0,14.480305,Q1-Graph4.csv
5,9.186047,72.0,14.257722,Q1-Graph5.csv
6,1.000000,1.0,0.000000,Q2-Seed1.csv
7,1.000000,1.0,0.000000,Q2-Seed2.csv
8,1.000000,1.0,0.000000,Q2-Seed3.csv


In [20]:
node_types = loaded["CGCS-GraphData-NodeTypes.csv"]

In [21]:
def node_type_distribution(df, node_types):
  if not {"Source", "Target"}.issubset(df.columns):
    return pd.Series(dtype=float)

  nodes = pd.DataFrame({
    "Node": list(set(df["Source"]) | set(df["Target"]))
  })

  merged = nodes.merge(node_types, left_on="Node", right_on="NodeID", how="left")

  return merged["NodeType"].value_counts(normalize=True)

node_type_stats = {}

for name, df in loaded.items():
  if {"Source", "Target"}.issubset(df.columns) and ("Graph" in name or "Template" in name):
    node_type_stats[name] = node_type_distribution(df, node_types)

pd.DataFrame(node_type_stats).fillna(0)

,CGCS-Template.csv,Q1-Graph1.csv,Q1-Graph2.csv,Q1-Graph3.csv,Q1-Graph4.csv,Q1-Graph5.csv
NodeType,,,,,,
1.0,0.000000,0.602151,0.563218,0.531646,0.448276,0.116279
2.0,0.033333,0.010753,0.011494,0.012658,0.149425,0.476744
3.0,0.000000,0.010753,0.045977,0.012658,0.000000,0.000000
4.0,0.966667,0.311828,0.333333,0.367089,0.333333,0.337209
5.0,0.000000,0.064516,0.045977,0.075949,0.068966,0.069767


In [22]:
def edge_coverage(df):
  return set(df["eType"].dropna().unique())

coverage = {}

for name, df in loaded.items():
  if ("Graph" in name or "Template" in name) and "eType" in df.columns:
    coverage[name] = edge_coverage(df)

coverage

{'CGCS-Template.csv': {np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(4),
  np.int64(5),
  np.int64(6)},
 'Q1-Graph1.csv': {np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(4),
  np.int64(5),
  np.int64(6)},
 'Q1-Graph2.csv': {np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(4),
  np.int64(5),
  np.int64(6)},
 'Q1-Graph3.csv': {np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(4),
  np.int64(5),
  np.int64(6)},
 'Q1-Graph4.csv': {np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(5),
  np.int64(6)},
 'Q1-Graph5.csv': {np.int64(0),
  np.int64(1),
  np.int64(2),
  np.int64(3),
  np.int64(5),
  np.int64(6)}}

In [23]:
template_types = coverage["CGCS-Template.csv"]

for name, types in coverage.items():
  if name != "CGCS-Template.csv":
    missing = template_types - types
    print(name, "missing:", missing)

Q1-Graph1.csv missing: set()
Q1-Graph2.csv missing: set()
Q1-Graph3.csv missing: set()
Q1-Graph4.csv missing: {np.int64(4)}
Q1-Graph5.csv missing: {np.int64(4)}


In [24]:
final = pd.DataFrame(stats).merge(
  pd.DataFrame(deg_stats),
  on="file"
)

final

,nodes,edges,density,file,avg_degree,max_degree,std_degree
0,88,1325,0.173067,CGCS-Template.csv,30.113636,208.0,34.151263
1,93,1216,0.142122,Q1-Graph1.csv,26.150538,135.0,21.401415
2,87,1300,0.173750,Q1-Graph2.csv,29.885057,192.0,27.735640
3,79,729,0.118306,Q1-Graph3.csv,18.455696,68.0,14.815790
4,87,732,0.097835,Q1-Graph4.csv,16.827586,64.0,14.480305
5,86,395,0.054036,Q1-Graph5.csv,9.186047,72.0,14.257722
6,2,1,0.500000,Q2-Seed1.csv,1.000000,1.0,0.000000
7,2,1,0.500000,Q2-Seed2.csv,1.000000,1.0,0.000000
8,2,1,0.500000,Q2-Seed3.csv,1.000000,1.0,0.000000


**Resumen del Análisis Estructural**

Entre los subgrafos candidatos, el Q1-Graph2 muestra la mayor similitud estructural con el grafo de referencia (template). Coincide estrechamente con la plantilla en términos de:

- número de nodos y aristas
- densidad del grafo
- distribución de grados
- composición de tipos de arista

En contraste, los Q1-Graph4 y Q1-Graph5 son estructuralmente incompletos, ya que carecen de aristas de coautoría (tipo 4), las cuales sí están presentes en la plantilla. Además, estos grafos presentan densidades más bajas y perfiles de interacción diferentes, lo que los convierte en candidatos poco probables.

In [25]:
SECONDS_PER_DAY = 86400

def add_time_features(df):
  df = df.copy()
  df["day"] = (df["Time"] // SECONDS_PER_DAY).astype(int)
  return df

In [26]:
graph_keys = [k for k in loaded if any(x in k for x in ["Graph", "Template"])]

temporal_data = {}

for name in graph_keys:
  df = loaded[name].copy()
  if "Time" in df.columns:
    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"])
    df["day"] = (df["Time"] // SECONDS_PER_DAY).astype(int)
    temporal_data[name] = df

In [27]:
daily_activity = {}

for name, df in temporal_data.items():
  daily = df.groupby("day").size()
  daily_activity[name] = daily

daily_df = pd.DataFrame(daily_activity).fillna(0)
daily_df.head()

,CGCS-Template.csv,Q1-Graph1.csv,Q1-Graph2.csv,Q1-Graph3.csv,Q1-Graph4.csv,Q1-Graph5.csv
day,,,,,,
-8481,0.0,0.0,1.0,0.0,0.0,0.0
-7663,0.0,1.0,0.0,0.0,0.0,0.0
-5748,0.0,0.0,1.0,0.0,0.0,0.0
-3215,0.0,0.0,1.0,0.0,0.0,0.0
-2428,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
def activity_by_etype(df):
  return df.groupby(["day", "eType"]).size().unstack(fill_value=0)

etype_time = {}

for name, df in temporal_data.items():
  etype_time[name] = activity_by_etype(df)


In [29]:
def detect_peaks(series):
  threshold = series.mean() + 2 * series.std()
  return series[series > threshold]

peaks = {}

for name, series in daily_activity.items():
  peaks[name] = detect_peaks(series)

In [31]:
template_series = daily_activity["CGCS-Template.csv"]

similarity = {}

for name, series in daily_activity.items():
  if name != "CGCS-Template.csv":
    aligned = pd.concat([template_series, series], axis=1).fillna(0)
    corr = aligned.corr().iloc[0,1]
    similarity[name] = corr

similarity

{'Q1-Graph1.csv': np.float64(0.9959923610910559),
 'Q1-Graph2.csv': np.float64(0.9951771308658628),
 'Q1-Graph3.csv': np.float64(0.9961137295700531),
 'Q1-Graph4.csv': np.float64(0.9953753983918113),
 'Q1-Graph5.csv': np.float64(0.9901174703457453)}

In [32]:
def shifted_correlation(a, b, max_lag=30):
  results = {}
  for lag in range(-max_lag, max_lag+1):
    shifted = b.shift(lag).fillna(0)
    corr = a.corr(shifted)
    results[lag] = corr
  return results

shift_results = {}

for name, series in daily_activity.items():
  if name != "CGCS-Template.csv":
    shift_results[name] = shifted_correlation(template_series, series)

In [33]:
best_shift = {}

for name, res in shift_results.items():
  best_lag = max(res, key=res.get)
  best_shift[name] = (best_lag, res[best_lag])

best_shift

{'Q1-Graph1.csv': (0, np.float64(0.9993371698536565)),
 'Q1-Graph2.csv': (0, np.float64(0.9978483908784472)),
 'Q1-Graph3.csv': (0, np.float64(0.9997024735648901)),
 'Q1-Graph4.csv': (0, np.float64(0.9990913489866791)),
 'Q1-Graph5.csv': (0, np.float64(0.9973359308610346))}

In [34]:
from scipy.spatial.distance import euclidean

distances = {}

for name, series in daily_activity.items():
  if name != "CGCS-Template.csv":
    aligned = pd.concat([template_series, series], axis=1).fillna(0)
    dist = euclidean(aligned.iloc[:,0], aligned.iloc[:,1])
    distances[name] = dist

distances

{'Q1-Graph1.csv': 169.08873410135874,
 'Q1-Graph2.csv': 151.1853167473614,
 'Q1-Graph3.csv': 183.21571984958058,
 'Q1-Graph4.csv': 207.3041244162788,
 'Q1-Graph5.csv': 491.8760006343062}